In [1]:
import sys
sys.path.append('/host/d/Github/')  # add the path to your own Example_UNet folder
import numpy as np
import nibabel as nb 
import pandas as pd
import os
import copy
import random
import Diffusion_denoising_thin_slice.simulation.CT.ct_basic_PCD as ct
import Diffusion_denoising_thin_slice.functions_collection as ff
import ct_projector.projector.numpy.parallel as ct_para
from skimage.metrics import structural_similarity
import lpips
import torch
## apt-get update && apt-get install -y libcufft10


main_path = '/host/e/D/Data/low_dose_CT/'  # change to your own data path

def split_angles_indexes(total_angles):
    arr = np.arange(total_angles)
   
    pairs = arr.reshape(-1, 2)  

    A = []
    B = []

    for pair in pairs:
        idx = random.randint(0, 1)  
        A.append(pair[idx])
        B.append(pair[1 - idx])

    A = np.array(A)
    B = np.array(B)
    return A,B

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# define the patient list
patient_sheet = ff.find_all_target_files(['nii_imgs/*'],main_path)
print('Found {} patients'.format(len(patient_sheet)))

Found 10 patients


In [3]:
def calc_mae_rmse_with_ref_window(img, ref, vmin, vmax):
    maes = []
    rmses = []
    for slice_num in range(0, img.shape[-1]):
        slice_img = img[:,:,slice_num]
        slice_ref = ref[:,:,slice_num]
        mask = np.where((slice_ref >= vmin) & (slice_ref <= vmax), 1, 0)
        # MAE
        mae = np.sum(np.abs(slice_img - slice_ref) * mask) / np.sum(mask)
        maes.append(mae)
        # RMSE
        rmse_value = np.sqrt( np.sum( (slice_img - slice_ref)**2 * mask) / np.sum(mask) )
        rmses.append(rmse_value)


    return np.mean(maes), np.std(maes), np.mean(rmses), np.std(rmses)

def calc_ssim_with_ref_window(img, ref, vmin, vmax):

    ssims = []
    for slice_num in range(0, img.shape[-1]):
        slice_img = img[:,:,slice_num]
        slice_ref = ref[:,:,slice_num]
        mask = np.where((slice_ref >= vmin) & (slice_ref <= vmax), 1, 0)
        _, ssim_map = structural_similarity(slice_img, slice_ref, data_range=vmax - vmin, full=True)
        ssim = np.sum(ssim_map * mask) / np.sum(mask)
        ssims.append(ssim)

    return np.mean(ssims), np.std(ssims)
def calc_lpips(imgs1, imgs2, vmin, vmax):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    loss_fn = lpips.LPIPS().to(device)
    
    lpipss = []
    for slice_num in range(0, imgs1.shape[-1]):
        slice1 = imgs1[:,:,slice_num]
        slice2 = imgs2[:,:,slice_num]

        slice1 = np.clip(slice1, vmin, vmax).astype(np.float32)
        slice2 = np.clip(slice2, vmin, vmax).astype(np.float32)

        slice1 = (slice1 - vmin) / (vmax - vmin) * 2 - 1
        slice2 = (slice2 - vmin) / (vmax - vmin) * 2 - 1

        slice1 = np.stack([slice1, slice1, slice1], axis=-1)
        slice2 = np.stack([slice2, slice2, slice2], axis=-1)
        # print('after stack, slice1 shape:', slice1.shape, ' slice2 shape:', slice2.shape)

        slice1 = np.transpose(slice1, (2, 0, 1))[np.newaxis, ...]
        slice2 = np.transpose(slice2, (2, 0, 1))[np.newaxis, ...]
        # print('after transpose, slice1 shape:', slice1.shape, ' slice2 shape:', slice2.shape)

        slice1 = torch.from_numpy(slice1).to(device)
        slice2 = torch.from_numpy(slice2).to(device)

        lpips_val = loss_fn(slice1, slice2)
        lpipss.append(lpips_val.item())

      

    return np.mean(lpipss), np.std(lpipss)


In [4]:
split_angle_recon = True
all_angle_recon = True
for i in range(0, len(patient_sheet)):
    patient_id = os.path.basename(patient_sheet[i])

    print(patient_id)

    save_folder_case = os.path.join(main_path,'simulation_v3', patient_id)
    ff.make_folder([os.path.dirname(save_folder_case), save_folder_case])


    # img file
    img_file = os.path.join(patient_sheet[i],'img.nii.gz')
    print(img_file)
    # load img
    img_clean = nb.load(img_file).get_fdata().astype(np.float32)
    img_clean[img_clean < -1024] = -1024
    spacing = nb.load(img_file).header.get_zooms()[::-1]
    affine = nb.load(img_file).affine
    print('img shape, min, max: ', img_clean.shape, np.min(img_clean), np.max(img_clean))
    print('spacing: ', spacing)

    for noise_type in [ 'poisson']:
        print('noise type: ', noise_type)
        # by vusialization we decided the following dose range
        # for MGH brain CT dataset
        # possion_hann_dose_range = [0.10,0.20]
        # gaussian_custom_dose_range = [0.15,0.25] 

        # for mayo low does CT dataset
        if noise_type[0:2] == 'po':
            dose_range = [0.10,0.20]#[0.10,0.20]
        elif noise_type[0:2] == 'ga':
            dose_range = [0.10,0.14]#[0.15, 0.26] 
        
        for k in range(0,1):
            save_folder_k = os.path.join(save_folder_case, noise_type+'_random_'+str(k));ff.make_folder([save_folder_k])

            # if os.path.isfile(os.path.join(save_folder_k,'recon_even.nii.gz')):
            #     print('already done, continue')
            #     continue

           
            if os.path.isfile(os.path.join(save_folder_k,'dose_factor.txt')):
            
                with open(os.path.join(save_folder_k,'dose_factor.txt'),'r') as f:
                    dose_factor = float(f.read())
                print('dose factor loaded: ', dose_factor)
            else:
                dose_factor = np.random.uniform(dose_range[0],dose_range[1] + 1e-8)
                # save into a txt file
                with open(os.path.join(save_folder_k,'dose_factor.txt'),'w') as f:
                    f.write('{:.4f}'.format(dose_factor))
            print('dose factor: ', dose_factor)


            # process img
            img0 = img_clean.copy()
            img0 = np.rollaxis(img0,-1,0)
      
            # define projectors
            geometry_file = '/host/d/Github/Diffusion_denoising_thin_slice/help_data/pcd_parallel_6x5_512.cfg'  # change to your own path
            projector = ct.define_forward_projector_pcd(img0,spacing, file_name = geometry_file)
            # FP
            # set angles
            angles = projector.get_angles()

            recon_all = np.zeros((img0.shape[1], img0.shape[2], img0.shape[0]), np.float32)
            recon_odd =  np.zeros((img0.shape[1], img0.shape[2], img0.shape[0]), np.float32)
            recon_even = np.zeros((img0.shape[1], img0.shape[2], img0.shape[0]), np.float32)
            # split angles
            odd_even_index_files = os.path.join('/host/e/D/Data/low_dose_CT/simulation_v2', patient_id, 'gaussian_random_0/odd_even_indexes.npy')
            if os.path.isfile(odd_even_index_files):
                print('load existing odd even indexes')
                odd_indexes = np.load(odd_even_index_files, allow_pickle=True).item()['odd']
                even_indexes = np.load(odd_even_index_files, allow_pickle=True).item()['even']
            else:
                odd_indexes, even_indexes = split_angles_indexes(len(angles))
                # save angles indexes for reference
                np.save(os.path.join(save_folder_k,'odd_even_indexes.npy'), {'odd':odd_indexes, 'even':even_indexes})

            for slice_n in range(0, img0.shape[0]):
                img_slice = img0[[slice_n],:,:].copy()
                img_slice = (img_slice[np.newaxis, ...] + 1000) / 1000 * 0.019 
               
                
                # forward proj
                prjs = ct_para.distance_driven_fp(projector, img_slice, angles)
                
                # add noise
                if noise_type[0:2] == 'po':
                    # add poisson noise
                    noise_of_prjs = ct.add_poisson_noise(prjs, N0=1000000, dose_factor = dose_factor) - prjs
                elif noise_type[0:2] == 'ga':
                    # add gaussian noise
                    noise_of_prjs = ct.add_gaussian_noise(prjs, N0=1000000, dose_factor = dose_factor) - prjs

                # recon
                if noise_type[0:2] == 'po':
                    # hann filter for noise projs
                    fnoise = ct_para.ramp_filter(projector, noise_of_prjs, 'hann')

                    # ramp filter for image
                    fprjs = ct_para.ramp_filter(projector, prjs, 'rl')

                    # recon all angles:
                    if all_angle_recon:
                        # bp for noise
                        recon_hann_noise = ct_para.distance_driven_bp(projector, fnoise, angles, True) 
                    
                        # bp for image
                        recon_hann_image = ct_para.distance_driven_bp(projector, fprjs, angles, True)
                    
                        # final recon
                        recon_hann = (recon_hann_noise +recon_hann_image)[0,0] / 0.019 * 1000 - 1000
                        
                        recon_all[:,:,slice_n] = recon_hann
                    
                    if split_angle_recon:
                        # split fnoise and fprjs to odd and even
                        fprjs_odd = fprjs[:, odd_indexes, ...]
                        fprjs_even = fprjs[:, even_indexes, ...]
                        fnoise_odd = fnoise[:, odd_indexes, ...]
                        fnoise_even = fnoise[:, even_indexes, ...]
                
                        # bp for image
                        recon_hann_image_odd = ct_para.distance_driven_bp(projector, fprjs_odd, angles[odd_indexes], True)
                        recon_hann_image_even = ct_para.distance_driven_bp(projector, fprjs_even, angles[even_indexes], True)

                        # bp for noise
                        recon_hann_noise_odd = ct_para.distance_driven_bp(projector, fnoise_odd, angles[odd_indexes], True)
                        recon_hann_noise_even = ct_para.distance_driven_bp(projector, fnoise_even, angles[even_indexes], True)

                        # final recon
                        recon_hann_odd = (recon_hann_image_odd + recon_hann_noise_odd) *2 / 0.019 * 1000 - 1000
                        recon_hann_even = (recon_hann_image_even + recon_hann_noise_even) *2 / 0.019 * 1000 - 1000
                        recon_odd[:,:,slice_n] = recon_hann_odd
                        recon_even[:,:,slice_n] = recon_hann_even
                        
                elif noise_type[0:2] == 'ga':

                    # custom filter
                    custom_filter_file = '/host/d/Github/Diffusion_denoising_thin_slice/help_data/softTissueKernel_65'  # change to your own path
                    custom_additional_filter = ct.get_additional_filter_to_rl(custom_filter_file, projector.nu, projector.du, projector.nview)

                    # ramp filter for image
                    fprjs, projector_new = ct.interleave_filter_and_recon(projector, prjs, custom_additional_filter, angles, get_recon=False, ramp_filter=True)
                    
                    # soft tissue kernel for noise
                    fnoise, _ = ct.interleave_filter_and_recon(projector, noise_of_prjs, custom_additional_filter, angles, get_recon=False , ramp_filter=False)
                    
                    if all_angle_recon:
                        # bp for image
                        recon_custom_image = ct_para.pixel_driven_bp(projector_new, fprjs, angles)
                        # bp for noise
                        recon_custom_noise = ct_para.pixel_driven_bp(projector_new, fnoise, angles)
                        # final recon
                        recon_custom = (recon_custom_noise + recon_custom_image)/ 0.019 * 1000 - 1000

                        recon_all[:,:,slice_n] = recon_custom
                    
                    if split_angle_recon:

                        # split fprjs and fnoise to odd and even
                        fprjs_odd = fprjs[:, odd_indexes, ...]
                        fprjs_even = fprjs[:, even_indexes, ...] 
                        fnoise_odd = fnoise[:, odd_indexes, ...]
                        fnoise_even = fnoise[:, even_indexes, ...]
                        
                        # # bp for image
                        recon_custom_image_odd = ct_para.pixel_driven_bp(projector_new, fprjs_odd, angles[odd_indexes])
                        recon_custom_image_even = ct_para.pixel_driven_bp(projector_new, fprjs_even, angles[even_indexes])

                        # # bp for noise
                        recon_custom_noise_odd = ct_para.pixel_driven_bp(projector_new, fnoise_odd, angles[odd_indexes])
                        recon_custom_noise_even = ct_para.pixel_driven_bp(projector_new, fnoise_even, angles[even_indexes])

                        # final recon
                        recon_custom_odd = (recon_custom_image_odd + recon_custom_noise_odd) *2  / 0.019 * 1000 - 1000
                        recon_custom_even = ( recon_custom_image_even + recon_custom_noise_even) *2 / 0.019 * 1000 - 1000

                        recon_odd[:,:,slice_n] = recon_custom_odd
                        recon_even[:,:,slice_n] = recon_custom_even
                        
            # calculate MAE, SSIM, LPIPS
            img_file = os.path.join(patient_sheet[i],'img.nii.gz')
            # load img
            img_clean = nb.load(img_file).get_fdata().astype(np.float32)
            vmin = -160
            vmax = 240
            if all_angle_recon:
                mae_all, mae_all_std, rmse_all, rmse_all_std = calc_mae_rmse_with_ref_window(recon_all, img_clean, vmin, vmax)
                ssim_all, ssim_all_std = calc_ssim_with_ref_window(recon_all, img_clean, vmin, vmax)
                lpips_all, lpips_all_std = calc_lpips(recon_all, img_clean, vmin, vmax)
                print('all angle recon, mae: {:.4f}'.format(mae_all), ' rmse: {:.4f}'.format(rmse_all), ' ssim: {:.4f}'.format(ssim_all), ' lpips: {:.4f}'.format(lpips_all))
            # if split_angle_recon:
            #     mae_odd, mae_odd_std, rmse_odd, rmse_odd_std = calc_mae_rmse_with_ref_window(recon_odd, img_clean, vmin, vmax)
            #     ssim_odd, ssim_odd_std = calc_ssim_with_ref_window(recon_odd, img_clean, vmin, vmax)
            #     lpips_odd, lpips_odd_std = calc_lpips(recon_odd, img_clean, vmin, vmax)
            #     # print('odd angle recon, mae: {:.4f}'.format(mae_odd), ' rmse: {:.4f}'.format(rmse_odd), ' ssim: {:.4f}'.format(ssim_odd), ' lpips: {:.4f}'.format(lpips_odd))

            #     mae_even, mae_even_std, rmse_even, rmse_even_std = calc_mae_rmse_with_ref_window(recon_even, img_clean, vmin, vmax)
            #     ssim_even, ssim_even_std = calc_ssim_with_ref_window(recon_even, img_clean, vmin, vmax)
            #     lpips_even, lpips_even_std = calc_lpips(recon_even, img_clean, vmin, vmax)
                # print('even angle recon, mae: {:.4f}'.format(mae_even), ' rmse: {:.4f}'.format(rmse_even), ' ssim: {:.4f}'.format(ssim_even), ' lpips: {:.4f}'.format(lpips_even))

            # save recon
            # nb.save(nb.Nifti1Image(img_clean, affine), os.path.join(save_folder_case,'img_clean.nii.gz'))
            if all_angle_recon:
                nb.save(nb.Nifti1Image(recon_all, affine), os.path.join(save_folder_k,'recon_all.nii.gz'))
                diff = recon_all - img_clean
                # nb.save(nb.Nifti1Image(diff, affine), os.path.join(save_folder_k,'diff_clean_all.nii.gz'))
            if split_angle_recon:
                nb.save(nb.Nifti1Image(recon_odd, affine), os.path.join(save_folder_k,'recon_odd.nii.gz'))
                nb.save(nb.Nifti1Image(recon_even, affine), os.path.join(save_folder_k,'recon_even.nii.gz'))
                diff_between = recon_odd - recon_even
                # nb.save(nb.Nifti1Image(diff_between, affine), os.path.join(save_folder_k,'diff_odd_even.nii.gz'))
            

L067
/host/e/D/Data/low_dose_CT/nii_imgs/L067/img.nii.gz
img shape, min, max:  (512, 512, 224) -1024.0 2123.0
spacing:  (2.0, 0.6640625, 0.6640625)
noise type:  poisson
dose factor:  0.18319745759886985
load existing odd even indexes
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
all angle recon, mae: 14.2630  rmse: 19.9428  ssim: 0.7958  lpips: 0.0589
L096
/host/e/D/Data/low_dose_CT/nii_imgs/L096/img.nii.gz
img shape, min, max:  (512, 512, 330) -1024.0 3071.0
spacing:  (2.0, 0.7421875, 0.7421875)
noise type:  poisson
dose factor:  0.17462816205847015
load existing odd even indexes
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
all angle recon, mae: 15.7206  rmse: 21.6464  ssim: 0.7206  lpips: 0.0948
L109
/host/e/D/Data/low_dose_CT/nii_imgs/L109/img.nii.gz
img shape, min, max:  (512, 512, 128) -1024.0 2220.0
spacing:  (2.0, 0.78125, 0.78125)
noise type:  poisson
dose factor:  0.15652823004729288
load existing odd even indexes
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


...

Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
all angle recon, mae: 16.9885  rmse: 28.5152  ssim: 0.6711  lpips: 0.1125
L333
/host/e/D/Data/low_dose_CT/nii_imgs/L333/img.nii.gz
img shape, min, max:  (512, 512, 244) -1024.0 1667.0
spacing:  (2.0, 0.78125, 0.78125)
noise type:  poisson
dose factor:  0.15384446911240482
load existing odd even indexes
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
all angle recon, mae: 17.7095  rmse: 24.4476  ssim: 0.7167  lpips: 0.0724
L506
/host/e/D/Data/low_dose_CT/nii_imgs/L506/img.nii.gz
img shape, min, max:  (512, 512, 211) -1024.0 3071.0
spacing:  (2.0, 0.7421875, 0.7421875)
noise type:  poisson
dose factor:  0.1888664519265643
load existing odd even indexes
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /usr/local/lib/python3.10/dist-packages/lpips/weights/v0.1/alex.pth
all angle recon, mae: 13.2392  rmse: 18.4008  ssim: 0.8386  lpips: 0.0343
